Go to http://jalammar.github.io/illustrated-bert/ and read about bert pretraining, heads.


In [ ]:
import torch
from transformers import BertModel, BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

text = "a nice walk by the river bank"
inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True)
outputs = model(**inputs, output_attentions=True)

attention_weights = outputs.attentions


In [ ]:
inputs

{'input_ids': tensor([[ 101, 1037, 3835, 3328, 2011, 1996, 2314, 2924,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [ ]:
len(attention_weights) # BERT has 12 layers

12

In [ ]:
attention_weights[0].shape # number of attention heads is 12BERT's attention mechanism is fascinating.
1, 12, 9, 9

NameError: name 'attention_weights' is not defined

In [ ]:
layer = 0
attention_head=0
attention_weights[layer][0, attention_head, :, :]

tensor([[0.0608, 0.1598, 0.0426, 0.0411, 0.1300, 0.1476, 0.0439, 0.0756, 0.2986],
        [0.1360, 0.0947, 0.1049, 0.1290, 0.0972, 0.1380, 0.0952, 0.1052, 0.0999],
        [0.0745, 0.0492, 0.1031, 0.1193, 0.1089, 0.0523, 0.1666, 0.1197, 0.2063],
        [0.0673, 0.0643, 0.2049, 0.0621, 0.0452, 0.1225, 0.1720, 0.1157, 0.1460],
        [0.1437, 0.0783, 0.1092, 0.1129, 0.1310, 0.0872, 0.0829, 0.0645, 0.1901],
        [0.1133, 0.0939, 0.1178, 0.0825, 0.1463, 0.1467, 0.1050, 0.0830, 0.1115],
        [0.0601, 0.0187, 0.1901, 0.2568, 0.0323, 0.0157, 0.1152, 0.2320, 0.0790],
        [0.1535, 0.0165, 0.2493, 0.1617, 0.0480, 0.0132, 0.1857, 0.0879, 0.0841],
        [0.1089, 0.1646, 0.0673, 0.0485, 0.1568, 0.1502, 0.0684, 0.0496, 0.1857]],
       grad_fn=<SliceBackward0>)

In [ ]:
model

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

# A Visual Notebook to Using BERT for the First TIme.ipynb



<img src="https://jalammar.github.io/images/distilBERT/bert-distilbert-sentence-classification.png" />

In this notebook, we will use pre-trained deep learning model to process some text. We will then use the output of that model to classify the text. The text is a list of sentences from film reviews. And we will calssify each sentence as either speaking "positively" about its subject of "negatively".

## Models: Sentence Sentiment Classification
Our goal is to create a model that takes a sentence (just like the ones in our dataset) and produces either 1 (indicating the sentence carries a positive sentiment) or a 0 (indicating the sentence carries a negative sentiment). We can think of it as looking like this:

<img src="https://jalammar.github.io/images/distilBERT/sentiment-classifier-1.png" />

Under the hood, the model is actually made up of two model.

* DistilBERT processes the sentence and passes along some information it extracted from it on to the next model. DistilBERT is a smaller version of BERT developed and open sourced by the team at HuggingFace. It’s a lighter and faster version of BERT that roughly matches its performance.
* The next model, a basic Logistic Regression model from scikit learn will take in the result of DistilBERT’s processing, and classify the sentence as either positive or negative (1 or 0, respectively).

The data we pass between the two models is a vector of size 768. We can think of this of vector as an embedding for the sentence that we can use for classification.


<img src="https://jalammar.github.io/images/distilBERT/distilbert-bert-sentiment-classifier.png" />

## Dataset
The dataset we will use in this example is [SST2](https://nlp.stanford.edu/sentiment/index.html), which contains sentences from movie reviews, each labeled as either positive (has the value 1) or negative (has the value 0):


<table class="features-table">
  <tr>
    <th class="mdc-text-light-green-600">
    sentence
    </th>
    <th class="mdc-text-purple-600">
    label
    </th>
  </tr>
  <tr>
    <td class="mdc-bg-light-green-50" style="text-align:left">
      a stirring , funny and finally transporting re imagining of beauty and the beast and 1930s horror films
    </td>
    <td class="mdc-bg-purple-50">
      1
    </td>
  </tr>
  <tr>
    <td class="mdc-bg-light-green-50" style="text-align:left">
      apparently reassembled from the cutting room floor of any given daytime soap
    </td>
    <td class="mdc-bg-purple-50">
      0
    </td>
  </tr>
  <tr>
    <td class="mdc-bg-light-green-50" style="text-align:left">
      they presume their audience won't sit still for a sociology lesson
    </td>
    <td class="mdc-bg-purple-50">
      0
    </td>
  </tr>
  <tr>
    <td class="mdc-bg-light-green-50" style="text-align:left">
      this is a visually stunning rumination on love , memory , history and the war between art and commerce
    </td>
    <td class="mdc-bg-purple-50">
      1
    </td>
  </tr>
  <tr>
    <td class="mdc-bg-light-green-50" style="text-align:left">
      jonathan parker 's bartleby should have been the be all end all of the modern office anomie films
    </td>
    <td class="mdc-bg-purple-50">
      1
    </td>
  </tr>
</table>

## Installing the transformers library
Let's start by installing the huggingface transformers library so we can load our deep learning NLP model.

In [ ]:
!pip install transformers --upgrade

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 59.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.5/224.5 kB 12.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 67.3 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import torch
import transformers as ppb
import warnings
warnings.filterwarnings('ignore')

## Importing the dataset
We'll use pandas to read the dataset and load it into a dataframe.

In [ ]:
df = pd.read_csv('https://github.com/clairett/pytorch-sentiment-classification/raw/master/data/SST2/train.tsv', delimiter='\t', header=None)

For performance reasons, we'll only use 2,000 sentences from the dataset

In [ ]:
batch_1 = df[:2000]

In [ ]:
df

,0,1
0,"a stirring , funny and finally transporting re...",1
1,apparently reassembled from the cutting room f...,0
2,they presume their audience wo n't sit still f...,0
3,this is a visually stunning rumination on love...,1
4,jonathan parker 's bartleby should have been t...,1
...,...,...
6915,"painful , horrifying and oppressively tragic ,...",1
6916,take care is nicely performed by a quintet of ...,0
6917,"the script covers huge , heavy topics in a bla...",0
6918,a seriously bad film with seriously warped log...,0


We can ask pandas how many sentences are labeled as "positive" (value 1) and how many are labeled "negative" (having the value 0)

In [ ]:
batch_1[1].value_counts()

1    1041
0     959
Name: 1, dtype: int64

## Loading the Pre-trained BERT model
Let's now load a pre-trained BERT model.

In [ ]:
# For DistilBERT:
model_class, tokenizer_class, pretrained_weights = (ppb.DistilBertModel, ppb.DistilBertTokenizer,
                                                    'distilbert-base-uncased')

## Want BERT instead of distilBERT? Uncomment the following line:
#model_class, tokenizer_class, pretrained_weights = (ppb.BertModel, ppb.BertTokenizer, 'bert-base-uncased')

# Load pretrained model/tokenizer
tokenizer = tokenizer_class.from_pretrained(pretrained_weights)
model = model_class.from_pretrained(pretrained_weights)

Some weights of the model checkpoint at distilbert-base-uncased were not used when initializing DistilBertModel: ['vocab_projector.bias', 'vocab_transform.weight', 'vocab_layer_norm.bias', 'vocab_layer_norm.weight', 'vocab_transform.bias', 'vocab_projector.weight']
- This IS expected if you are initializing DistilBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DistilBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
model

DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): MultiHeadSelfAttention(
          (dropout): Dropout(p=0.1, inplace=False)
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): Li

Right now, the variable `model` holds a pretrained distilBERT model -- a version of BERT that is smaller, but much faster and requiring a lot less memory.

## Model #1: Preparing the Dataset
Before we can hand our sentences to BERT, we need to so some minimal processing to put them in the format it requires.

### Tokenization
Our first step is to tokenize the sentences -- break them up into word and subwords in the format BERT is comfortable with.

In [ ]:
tokenized = batch_1[0].apply((lambda x: tokenizer.encode(x, add_special_tokens=True)))

In [ ]:
tokenized[14], tokenizer.decode(tokenized[11])

([101,
  2089,
  2022,
  2062,
  8991,
  4818,
  2084,
  13749,
  18595,
  3560,
  1010,
  2021,
  2009,
  4152,
  1996,
  3105,
  2589,
  102],
 "[CLS] the most repugnant adaptation of a classic text since roland joff and demi moore's the scarlet letter [SEP]")

In [ ]:
tokenizer.encode(batch_1[0][10], add_special_tokens=True)

[101,
 1037,
 16376,
 19240,
 2005,
 3451,
 1998,
 3167,
 2969,
 5456,
 1998,
 1037,
 27263,
 17933,
 4226,
 3193,
 1997,
 1037,
 2210,
 4622,
 2088,
 102]

In [ ]:
batch_1[0][11]

"the most repugnant adaptation of a classic text since roland joff and demi moore 's the scarlet letter"

<img src="https://jalammar.github.io/images/distilBERT/bert-distilbert-tokenization-2-token-ids.png" />

### Padding
After tokenization, `tokenized` is a list of sentences -- each sentences is represented as a list of tokens. We want BERT to process our examples all at once (as one batch). It's just faster that way. For that reason, we need to pad all lists to the same size, so we can represent the input as one 2-d array, rather than a list of lists (of different lengths).

In [ ]:
max_len = 0
for i in tokenized.values:
    if len(i) > max_len:
        max_len = len(i)
padded = np.array([i + [0]*(max_len-len(i)) for i in tokenized.values])


In [ ]:
padded[1]

array([  101,  4593,  2128, 27241, 23931,  2013,  1996,  6276,  2282,
        2723,  1997,  2151,  2445, 12217,  7815,   102,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0])

Our dataset is now in the `padded` variable, we can view its dimensions below:

In [ ]:
np.array(padded).shape

(2000, 59)

### Masking
If we directly send `padded` to BERT, that would slightly confuse it. We need to create another variable to tell it to ignore (mask) the padding we've added when it's processing its input. That's what attention_mask is:

In [ ]:
attention_mask = np.where(padded != 0, 1, 0)
attention_mask.shape
attention_mask[2], padded[2]

(array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 array([  101,  2027,  3653, 23545,  2037,  4378, 24185,  1050,  1005,
         1056,  4133,  2145,  2005,  1037, 11507, 10800,  1010,  2174,
        14036,  2135,  3591,  1010,  2061,  2027, 19817,  4140,  2041,
         1996,  7511,  2671,  4349,  3787,  1997, 11829,  7168,  9219,
         1998, 28971,  2308,  1999,  8301,  8737,  2100,  4253,   102,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0]))

## Model #1: And Now, Deep Learning!
Now that we have our model and inputs ready, let's run our model!

<img src="https://jalammar.github.io/images/distilBERT/bert-distilbert-tutorial-sentence-embedding.png" />

The `model()` function runs our sentences through BERT. The results of the processing will be returned into `last_hidden_states`.

In [ ]:
input_ids = torch.tensor(padded)
attention_mask = torch.tensor(attention_mask)

with torch.no_grad():
   outputs = model(input_ids, attention_mask=attention_mask)
help(outputs)

Help on BaseModelOutput in module transformers.modeling_outputs object:

class BaseModelOutput(transformers.utils.generic.ModelOutput)
 |  BaseModelOutput(last_hidden_state: torch.FloatTensor = None, hidden_states: Optional[Tuple[torch.FloatTensor]] = None, attentions: Optional[Tuple[torch.FloatTensor]] = None) -> None
 |  
 |  Base class for model's outputs, with potential hidden states and attentions.
 |  
 |  Args:
 |      last_hidden_state (`torch.FloatTensor` of shape `(batch_size, sequence_length, hidden_size)`):
 |          Sequence of hidden-states at the output of the last layer of the model.
 |      hidden_states (`tuple(torch.FloatTensor)`, *optional*, returned when `output_hidden_states=True` is passed or when `config.output_hidden_states=True`):
 |          Tuple of `torch.FloatTensor` (one for the output of the embeddings, if the model has an embedding layer, +
 |          one for the output of each layer) of shape `(batch_size, sequence_length, hidden_size)`.
 |  
 |    

In [ ]:
outputs.last_hidden_state.numpy().shape
#outputs is called in the original notebook last_hidden_states
#last_hidden_state (torch.FloatTensor of shape (batch_size, sequence_length, hidden_size))
            # a Sequence of hidden-states at the output of the last layer of the model.

(2000, 59, 768)

Let's slice only the part of the output that we need. That is the output corresponding the first token of each sentence. The way BERT does sentence classification, is that it adds a token called `[CLS]` (for classification) at the beginning of every sentence. The output corresponding to that token can be thought of as an embedding for the entire sentence.

<img src="https://jalammar.github.io/images/distilBERT/bert-output-tensor-selection.png" />

We'll save those in the `features` variable, as they'll serve as the features to our logitics regression model.

In [ ]:
features = outputs.last_hidden_state[:,0,:].numpy()

In [ ]:
features[2]

array([-5.06335422e-02,  7.20394701e-02, -2.95973364e-02, -1.40354440e-01,
       -8.06072578e-02, -1.03232235e-01,  4.26805198e-01,  2.45677799e-01,
        4.69817929e-02, -1.48649126e-01, -1.25710443e-01, -1.91825465e-03,
        9.26228985e-03,  3.26775342e-01, -6.11782912e-03,  4.95377451e-01,
       -6.87092468e-02,  1.56111658e-01,  2.09620714e-01, -7.67024159e-02,
        2.29849786e-01, -2.94079781e-01, -6.16846234e-02,  1.86952829e-01,
        9.29776654e-02, -1.10866748e-01,  1.28718749e-01,  1.06657110e-01,
       -3.78832826e-03,  8.16244483e-02,  9.48646814e-02,  1.63880482e-01,
        1.62678175e-02, -3.22637439e-01,  1.60817672e-02, -9.81822014e-02,
        7.04560652e-02, -1.36364298e-02,  1.91389620e-01,  1.41441017e-01,
        1.14794791e-01,  1.06009059e-01,  2.02923253e-01,  5.09373546e-02,
       -5.49217649e-02, -3.81684840e-01, -2.59828091e+00, -1.97298735e-01,
       -4.09966022e-01, -2.61619031e-01,  2.55241632e-01,  1.81375220e-01,
        2.37444192e-01,  

The labels indicating which sentence is positive and negative now go into the `labels` variable

In [ ]:
labels = batch_1[1]

## Model #2: Train/Test Split
Let's now split our datset into a training set and testing set (even though we're using 2,000 sentences from the SST2 training set).

In [ ]:
train_features, test_features, train_labels, test_labels = train_test_split(features, labels)

<img src="https://jalammar.github.io/images/distilBERT/bert-distilbert-train-test-split-sentence-embedding.png" />

### [Bonus] Grid Search for Parameters
We can dive into Logistic regression directly with the Scikit Learn default parameters, but sometimes it's worth searching for the best value of the C parameter, which determines regularization strength.

In [ ]:
# parameters = {'C': np.linspace(0.0001, 100, 20)}
# grid_search = GridSearchCV(LogisticRegression(), parameters)
# grid_search.fit(train_features, train_labels)

# print('best parameters: ', grid_search.best_params_)
# print('best scrores: ', grid_search.best_score_)

We now train the LogisticRegression model. If you've chosen to do the gridsearch, you can plug the value of C into the model declaration (e.g. `LogisticRegression(C=5.2)`).

In [ ]:
lr_clf = LogisticRegression()
lr_clf.fit(train_features, train_labels)

LogisticRegression()

<img src="https://jalammar.github.io/images/distilBERT/bert-training-logistic-regression.png" />

## Evaluating Model #2
So how well does our model do in classifying sentences? One way is to check the accuracy against the testing dataset:

In [ ]:
lr_clf.score(test_features, test_labels)

0.824

How good is this score? What can we compare it against? Let's first look at a dummy classifier:

In [ ]:
from sklearn.dummy import DummyClassifier
clf = DummyClassifier()

scores = cross_val_score(clf, train_features, train_labels)
print("Dummy classifier score: %0.3f (+/- %0.2f)" % (scores.mean(), scores.std() * 2))

Dummy classifier score: 0.515 (+/- 0.00)


So our model clearly does better than a dummy classifier. But how does it compare against the best models?

## Proper SST2 scores
For reference, the [highest accuracy score](http://nlpprogress.com/english/sentiment_analysis.html) for this dataset is currently **96.8**. DistilBERT can be trained to improve its score on this task – a process called **fine-tuning** which updates BERT’s weights to make it achieve a better performance in this sentence classification task (which we can call the downstream task). The fine-tuned DistilBERT turns out to achieve an accuracy score of **90.7**. The full size BERT model achieves **94.9**.



And that’s it! That’s a good first contact with BERT. The next step would be to head over to the documentation and try your hand at [fine-tuning](https://huggingface.co/transformers/examples.html#glue). You can also go back and switch from distilBERT to BERT and see how that works.

### Let's try with fine tunning
We will first tokenize the data and then use Trainer for fine tunning of Bert

In [ ]:

!pip uninstall -y transformers accelerate
!pip install transformers accelerate datasets

Found existing installation: transformers 4.29.2
Uninstalling transformers-4.29.2:
  Successfully uninstalled transformers-4.29.2
Found existing installation: accelerate 0.19.0
Uninstalling accelerate-0.19.0:
  Successfully uninstalled accelerate-0.19.0
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
  Using cached transformers-4.29.2-py3-none-any.whl (7.1 MB)
  Using cached accelerate-0.19.0-py3-none-any.whl (219 kB)


In [ ]:
import pandas as pd
df = pd.read_csv('https://github.com/clairett/pytorch-sentiment-classification/raw/master/data/SST2/train.tsv', delimiter='\t', header=None)

In [ ]:
#tokenizer.decode(input_ids[0])

In [ ]:
#let's work with datasets
from datasets import load_dataset
from datasets import Dataset

df=df.rename(columns={0: 'text', 1: 'label'})

In [ ]:
df

,text,label
0,"a stirring , funny and finally transporting re...",1
1,apparently reassembled from the cutting room f...,0
2,they presume their audience wo n't sit still f...,0
3,this is a visually stunning rumination on love...,1
4,jonathan parker 's bartleby should have been t...,1
...,...,...
6915,"painful , horrifying and oppressively tragic ,...",1
6916,take care is nicely performed by a quintet of ...,0
6917,"the script covers huge , heavy topics in a bla...",0
6918,a seriously bad film with seriously warped log...,0


In [ ]:

dataset = Dataset.from_pandas(df)
splitted  = dataset.train_test_split(test_size=0.25)


In [ ]:
splitted

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 5190
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1730
    })
})

In [ ]:
splitted['train'], splitted['test']

(Dataset({
     features: ['text', 'label'],
     num_rows: 5190
 }),
 Dataset({
     features: ['text', 'label'],
     num_rows: 1730
 }))

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", max_length=100, truncation=True)

tokenized_splitted = splitted.map(tokenize_function, batched=True, batch_size=32)
#tokenized_dataset_test = dataset_test.map(tokenize_function, batched=True)

Map:   0%|          | 0/5190 [00:00<?, ? examples/s]

Map:   0%|          | 0/1730 [00:00<?, ? examples/s]

In [ ]:
tokenized_splitted

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 5190
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 1730
    })
})

In [ ]:
tokenized_splitted['train'][23]['input_ids'], tokenized_splitted['train'][1]['label']

([101,
  1996,
  3185,
  2003,
  2471,
  3294,
  11158,
  1999,
  23873,
  1010,
  4474,
  1998,
  8335,
  6832,
  10652,
  102,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 1)

In [ ]:
len(tokenized_splitted['train'][1]['attention_mask']), len(tokenized_splitted['train'][1]['attention_mask'])

(100, 100)

In [ ]:
from transformers import AutoModelForSequenceClassification

model1 = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

Some weights of the model checkpoint at distilbert-base-uncased were not used when initializing DistilBertForSequenceClassification: ['vocab_projector.bias', 'vocab_transform.weight', 'vocab_transform.bias', 'vocab_projector.weight', 'vocab_layer_norm.weight', 'vocab_layer_norm.bias']
- This IS expected if you are initializing DistilBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DistilBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'pre_classifier.weight', 'classifier

In [ ]:
model1
#sum(p.numel() for p in model1.parameters() if p.requires_grad)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
 

In [ ]:
import torch
#device = torch.device("cpu") if torch.cuda.is_available() else torch.device("cuda")
#model1.to(device)
#device

In [ ]:
#default arguments
from transformers import TrainingArguments, Trainer

#training_args = TrainingArguments("test_trainer")



In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import accuracy_score
import numpy as np
torch.device("cpu")
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }


training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=200,
    weight_decay=0.01,
    evaluation_strategy="steps",
    logging_strategy="steps",
    eval_steps=50,
    logging_dir='./logs',
    logging_steps = 50

)

trainer = Trainer(
    model=model1,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=tokenized_splitted['train'],
    eval_dataset=tokenized_splitted['test'],
    #place_model_on_device=True
)

In [ ]:
#from transformers import Trainer

#trainer = Trainer(
#    model=model, args=training_args, train_dataset=tokenized_splitted['train'],
#    eval_dataset=tokenized_splitted['test']
#)

In [ ]:
trainer.train()

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:407: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
50,0.684800,0.644849,0.667052,0.732342,0.631917,0.870718
100,0.485600,0.334933,0.861850,0.858997,0.921519,0.804420
150,0.354700,0.318916,0.873410,0.880914,0.867238,0.895028
200,0.237300,0.390165,0.872832,0.871043,0.927591,0.820994
250,0.249900,0.285724,0.886127,0.890495,0.895973,0.885083
300,0.220100,0.285602,0.888439,0.892359,0.900901,0.883978
350,0.192700,0.339552,0.893642,0.896513,0.912944,0.880663
400,0.077800,0.352296,0.889595,0.892877,0.906606,0.879558
450,0.081600,0.376976,0.885549,0.891209,0.886339,0.896133


TrainOutput(global_step=489, training_loss=0.27013303068274125, metrics={'train_runtime': 171.8627, 'train_samples_per_second': 90.596, 'train_steps_per_second': 2.845, 'total_flos': 402835429116000.0, 'train_loss': 0.27013303068274125, 'epoch': 3.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 0.38103818893432617,
 'eval_accuracy': 0.8867052023121387,
 'eval_f1': 0.8919514884233738,
 'eval_precision': 0.88998899889989,
 'eval_recall': 0.8939226519337017,
 'eval_runtime': 5.145,
 'eval_samples_per_second': 336.249,
 'eval_steps_per_second': 10.69,
 'epoch': 3.0}

In [ ]:
#%load_ext tensorboard
#%tensorboard --logdir logs

In [ ]:
sentence="I am so tired and I still do not know if it works."
sentence="I am not sad."
#sentence="We were happy after the work was done."
#sentence = "It's winter and it  is  snowing outside"
#sentence = "It's winter and it is snowing outside"
tokenized = tokenizer(sentence, padding="max_length", max_length=100, truncation=True)

In [ ]:
preds=trainer.predict([tokenized])
preds

PredictionOutput(predictions=array([[-2.078637 ,  1.7175981]], dtype=float32), label_ids=None, metrics={'test_runtime': 0.0361, 'test_samples_per_second': 27.703, 'test_steps_per_second': 27.703})

In [ ]:
import tensorflow as tf
tf_predictions = tf.nn.softmax(preds.predictions, axis=-1)
tf_predictions.numpy()

array([[0.02196199, 0.97803795]], dtype=float32)

In [ ]:
#torch.cuda.empty_cache()

In [ ]:
from transformers import BartTokenizer, BartForCausalLM

import torch
#sentence = "It's winter and it is snowing outside"
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import GPTNeoForCausalLM, GPT2Tokenizer

#tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
#model = GPT2LMHeadModel.from_pretrained('gpt2')


tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPTNeoForCausalLM.from_pretrained('EleutherAI/gpt-neo-1.3B')

inputs = tokenizer.encode("Hello, my dog is cute", return_tensors="pt")
outputs = model.generate(inputs, labels=inputs, do_sample=True, temperature=0.9, max_length=100)
tokenizer.decode(outputs[0])

#last_hidden_states = outputs.last_hidden_state


In [ ]:
from transformers import pipeline
generator = pipeline('text-generation', model='EleutherAI/gpt-neo-1.3B')
generator("EleutherAI has", do_sample=True, min_length=50)

In [ ]:
# set return_num_sequences > 1
beam_outputs = model.generate(
    input_ids,
    max_length = 20,
    num_beams = 5,
    no_repeat_ngram_size = 2,
    num_return_sequences = 5,
    early_stopping = True
)

In [ ]:
for i, beam_output in enumerate(beam_outputs):
      print("{}: {}".format(i, tokenizer.decode(beam_output, skip_special_tokens=True)))


In [ ]:
from transformers import DistilBertModel, DistilBertForSequenceClassification,  DistilBertForTokenClassification, DistilBertForMaskedLM
#tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = DistilBertModel.from_pretrained('distilbert-base-uncased')
model1 = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased')
model2 = DistilBertForTokenClassification.from_pretrained('distilbert-base-uncased')
model3 = DistilBertForMaskedLM.from_pretrained('distilbert-base-uncased')

In [ ]:
model

In [ ]:
model1

In [ ]:
model2

In [ ]:
model3

In [ ]:
from transformers import BartModel, BartForSequenceClassification,  BartForConditionalGeneration
#tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model10 = BartModel.from_pretrained('facebook/bart-large')
model11= BartForSequenceClassification.from_pretrained('facebook/bart-large')
model12 = BartForConditionalGeneration.from_pretrained('facebook/bart-large')


In [ ]:
model10

In [ ]:
model11

In [ ]:
model12